In [1]:
#import packages
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely
import random
from numpy import random
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import glob
from scipy.stats import norm
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from scipy.interpolate import griddata
import math
import random
import os
import sys
# from rasterstats import zonal_stats
import rasterio
import json
import zarr

In [15]:
plot_folder = '/field_data_ucca_08_2025/'
site_name = 'puc'
gui_folder = '/puc_test/'
project_folder = '/pacific_union_college/'
avg_tpa = None

ff_api_key = 'api-key'
lfm = 100

tree_species_code = pd.read_csv('/FIATreeSpeciesCode.csv')


In [7]:

os.environ["FASTFUELS_API_KEY"] = ff_api_key

from fastfuels_sdk.domains import Domain
from fastfuels_sdk.features import Features
from fastfuels_sdk.grids import TopographyGridBuilder
from fastfuels_sdk.inventories import Inventories
from fastfuels_sdk.grids import TreeGridBuilder
from fastfuels_sdk.grids import SurfaceGridBuilder
from fastfuels_sdk.grids import Grids

In [8]:

def RSE(y_true, y_predicted):
    """
    root mean square error
    - y_true: Actual values
    - y_predicted: Predicted values
    """
    y_true = np.array(y_true)
    y_predicted = np.array(y_predicted)
    RSS = np.sum(np.square(y_true - y_predicted))
     
    rse = math.sqrt(RSS / (len(y_true) - 2))
    return rse

def lin_reg(df, x_name, y_name, predict_x, simulate = False):
    '''
    linear regression predictions
    df: dataframe
    x_name: column name for x variable
    y_name: column name for y variable
    predict_x: x variable that you want to predict on
    simulate: true or false, adds noise to the prediction
    '''
    df = df[[x_name, y_name]].dropna()
     
    X = np.array(df[x_name]).reshape(np.array(df[x_name]).shape[0],1)
    Y = np.array(df[y_name]).reshape(np.array(df[y_name]).shape[0],1)
    
    mod = LinearRegression()
    mod_fit = mod.fit(X,Y)
     
    predictions_true = mod.predict(X)
    predictions = mod.predict(predict_x)
     
    if simulate:
        sd = RSE(Y, predictions_true)
        simulations = [np.random.normal(d, sd)[0] for d in predictions]
        return(simulations)
     
    else:
        predictions = [d[0] for d in predictions]
        return(predictions)

def create_domain_and_grids(boundary, site_name, product, 
                            version, interpolation_method, 
                            curing_live_herbaceous, curing_live_woody, 
                            groups, feature_masks, remove_non_burnable, 
                            uniform_fuel_moisture_value, grid_data_zip_path,
                            tree_inventory = False, tree_inventory_path = None,
                            horizontal_resolution=2.0, vertical_resolution=1.0):
     
    """
    Create a domain and associated grids for a given site in fastfuels.
     
    Parameters:
    boundary (GeoDataFrame): Geospatial boundary of the site.
    site_name (str): Name of the site.
    product (str): Product type for fuel load data.
    version (str): Version of the product.
    interpolation_method (str): Method for interpolating data.
    curing_live_herbaceous (float): Curing percentage for live herbaceous fuels.
    curing_live_woody (float): Curing percentage for live woody fuels.
    groups (list): List of groups for fuel data.
    feature_masks (dict): Dictionary of feature masks.
    remove_non_burnable (bool): Flag to remove non-burnable areas.
    uniform_fuel_moisture_value (float): Uniform fuel moisture value.
    grid_data_zip_path (str): Path to save grid_data
    tree_inventory (bool): Flag to make tree inventory out of csv file
    tree_inventory_path (str): path to tree inventory csv
    horizontal_resolution (float, optional): Horizontal resolution for the domain. Default is 2.0.
    vertical_resolution (float, optional): Vertical resolution for the domain. Default is 1.0.
     
    Returns:
    dict: A dictionary containing the created domain, features, grids, feature grid, topography grid, tree inventory, tree grid, and surface grid.
    """
    
    # Create domain
    domain = Domain.from_geodataframe(
        geodataframe=boundary,
        name=site_name,
        description=f"domain of {site_name}",
        horizontal_resolution=horizontal_resolution,  # Use the argument
        vertical_resolution=vertical_resolution       # Use the argument
    )
    print(f"Created domain with ID: {domain.id}")
     
    # Initialize Features for our domain
    features = Features.from_domain_id(domain.id)
     
    # Initialize grids for our domain
    grids = Grids.from_domain_id(domain.id)
     
    # Create features from OpenStreetMap
    road_feature = features.create_road_feature_from_osm()
    water_feature = features.create_water_feature_from_osm()
     
    # Wait for features to be ready
    road_feature.wait_until_completed()
    water_feature.wait_until_completed()
     
    print("created road and water")
     
    # Create a feature grid with roads and water bodies
    feature_grid = grids.create_feature_grid(
        attributes=["road", "water"]
    )
    feature_grid.wait_until_completed()
    print("created feature grid")
     
    topography_grid = (
        TopographyGridBuilder(domain_id=domain.id)
        .with_elevation_from_landfire(interpolation_method="linear")
        #.with_elevation_from_3dep(interpolation_method="linear")
        .build()
    )
     
    topography_grid.wait_until_completed()
    print("created topo")
     
    # if tree inventory
    if tree_inventory:
        tree_inventory = Inventories.from_domain_id(
            domain.id
        ).create_tree_inventory_from_file_upload(
            # feature_masks=feature_masks,
            file_path=(tree_inventory_path)
        )
        tree_inventory.wait_until_completed()
        print("created tree")
     
    # else Create tree inventory
    else:
        tree_inventory = Inventories.from_domain_id(
            domain.id
        ).create_tree_inventory_from_treemap(
            feature_masks=feature_masks
        )
        tree_inventory.wait_until_completed()
        print("created tree")
    
     
    # Create tree grid
    tree_grid = (
        TreeGridBuilder(domain_id=domain.id)
        .with_bulk_density_from_tree_inventory()
        .with_spcd_from_tree_inventory()
        .with_uniform_fuel_moisture(lfm)
        .build()
    )
    tree_grid.wait_until_completed()
     
    print("created treegrid")
     
    # Create surface grid
    surface_grid = (
        SurfaceGridBuilder(domain_id=domain.id)
        .with_fuel_load_from_landfire(
            product=product,
            version=version,
            interpolation_method=interpolation_method,
            curing_live_herbaceous=curing_live_herbaceous,
            curing_live_woody=curing_live_woody,
            groups=groups,
            feature_masks=feature_masks,
            remove_non_burnable=remove_non_burnable,
        )
        .with_fuel_depth_from_landfire(
            product=product,
            version=version,
            interpolation_method=interpolation_method,
            feature_masks=feature_masks,
            remove_non_burnable=remove_non_burnable,
        )
        # .with_uniform_fuel_moisture(
        #     value=uniform_fuel_moisture_value,
        #     feature_masks=feature_masks
        # )
        .with_uniform_fuel_moisture_by_size_class(     
            one_hour=uniform_fuel_moisture_value * 100,
            ten_hour=15.0,
            hundred_hour=20.0,
            live_herbaceous=lfm,
            live_woody=lfm,
            feature_masks=["road", "water"]
        )
        .with_savr_from_landfire(
            product=product,
            version=version,
            interpolation_method=interpolation_method,
            groups=groups,
            feature_masks=feature_masks,
            remove_non_burnable=remove_non_burnable
        )
        .build()
    )
    surface_grid.wait_until_completed()
    print("created surface")
     
    # Create the export
    export = grids.create_export("zarr") #"zarr" or "QUICFire"
         
    # Wait for the export to complete
    export.wait_until_completed()
    print('exported')
    
    # Save to a file
    export.to_file(grid_data_zip_path)
     
     
    return {
        "domain": domain,
        "features": features,
        "grids": grids,
        "feature_grid": feature_grid,
        "topography_grid": topography_grid,
        "tree_inventory": tree_inventory,
        "tree_grid": tree_grid,
        "surface_grid": surface_grid
    }

def calculate_slope_manual(dtm_array, cell_size):
    # Calculate gradients in x and y directions
    dz_dx = np.gradient(dtm_array, axis=1) / cell_size
    dz_dy = np.gradient(dtm_array, axis=0) / cell_size

    # Calculate slope in radians
    slope_radians = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))

    # Convert to degrees
    slope_degrees = np.degrees(slope_radians)
    return slope_degrees

def calculate_aspect(dem_array, cell_size):
    """
    Calculates the aspect from a Digital Elevation Model (DEM) array.

    Args:
        dem_array (np.ndarray): A 2D NumPy array representing the DEM.
        cell_size (float): The spatial resolution of the DEM (e.g., in meters).

    Returns:
        np.ndarray: A 2D NumPy array representing the aspect in degrees (0-360, North at 0/360).
                    Flat areas are represented by NaN.
    """
    rows, cols = dem_array.shape
    aspect = np.full(dem_array.shape, np.nan)

    # Pad the DEM to handle edges, or iterate only over inner pixels
    # For simplicity, this example iterates over inner pixels
    for r in range(1, rows - 1):
        for c in range(1, cols - 1):
            # 3x3 window
            window = dem_array[r-1:r+2, c-1:c+2]

            # Calculate dz/dx and dz/dy using Horn's method
            dz_dx = ((window[0, 2] + 2*window[1, 2] + window[2, 2]) -
                     (window[0, 0] + 2*window[1, 0] + window[2, 0])) / (8 * cell_size)

            dz_dy = ((window[2, 0] + 2*window[2, 1] + window[2, 2]) -
                     (window[0, 0] + 2*window[0, 1] + window[0, 2])) / (8 * cell_size)

            # Handle flat areas
            if dz_dx == 0 and dz_dy == 0:
                aspect[r, c] = np.nan  # Or a specific value like -1
            else:
                aspect_rad = np.arctan2(dz_dy, -dz_dx)
                aspect_deg = np.degrees(aspect_rad)

                # Convert to compass bearings (0-360, North at 0/360)
                if aspect_deg < 0:
                    aspect_deg = 90 - aspect_deg
                else:
                    aspect_deg = 90 - aspect_deg
                    if aspect_deg < 0:
                        aspect_deg += 360

                aspect[r, c] = aspect_deg
    return aspect


In [10]:
#read in the boundary
boundary = gpd.read_file(gui_folder + 'ff_domain.geojson')

als_trees = gpd.read_file(project_folder + 'data/treetops.geojson')
# als_trees = als_trees.set_crs(proj, allow_override = True)
boundary = boundary.to_crs(als_trees.crs)

tree = '04_tree.csv'

In [11]:

#read in als tree list
als_trees = gpd.read_file(project_folder + 'data/treetops.geojson')

#make sure the als trees are in the boundary
als_trees_in_boundary = als_trees.sjoin(boundary)
als_trees_in_boundary = als_trees_in_boundary.drop(['index_right'], axis=1)

#try to calculate how many trees to add based on the number of trees in the als file

boundary_area_acres = boundary.area/4047
als_tpa = (als_trees_in_boundary.shape[0] / boundary_area_acres)[0]

if avg_tpa is None:
    avg_tpa = als_tpa * 1.5


In [12]:

tree = '04_tree.csv'
# site_name = 'ind'
field_trees = pd.read_csv(plot_folder + tree)
plots_df = pd.read_csv(plot_folder + '01_plot_identification.csv')

plots_df = plots_df[plots_df.site_name == site_name]
field_trees = field_trees.merge(plots_df, on = 'inventory_id')

n_plots = plots_df.shape[0] 
tpa_list = []
plot_size = (1/10) 

for i in np.unique(field_trees.inventory_id):
    ft_sub = field_trees[field_trees.inventory_id == i]
    n_trees = ft_sub.shape[0]

    tpa_list.append(n_trees * 10)

avg_tpa = np.median(np.array(tpa_list))

#calculate the slope and aspect
# Load the DTM
with rasterio.open(project_folder + "data/dtm.tif") as src:
    dtm_array = src.read(1)
    # Assuming square pixels, get cell size from transform
    cell_size = src.res[0]

# fix DTM and holes in DTM
dtm_array[dtm_array == -9999] = np.nan

image_with_nan = dtm_array
# Get the coordinates of valid (non-NaN) points
valid_coords = np.argwhere(~np.isnan(image_with_nan))
valid_values = image_with_nan[~np.isnan(image_with_nan)]

# Get the coordinates of all points in the image (including NaNs)
all_coords = np.indices(image_with_nan.shape).reshape(2, -1).T

# Interpolate the NaN values
# 'linear' is a common method; 'nearest' and 'cubic' are also options
interpolated_image = griddata(valid_coords, valid_values, all_coords, method='linear')

# Reshape the interpolated data back to the original image shape
filled_image = interpolated_image.reshape(image_with_nan.shape)

dtm_filled = filled_image

# Calculate slope
slope_map = calculate_slope_manual(dtm_filled, cell_size)

# calcualte aspect
aspect_map = calculate_aspect(dtm_filled, cell_size)


In [13]:

#calculate how many trees to add
trees_to_add = round((boundary_area_acres * avg_tpa) - als_trees_in_boundary.shape[0])[0]
print(trees_to_add)

#bin the als trees, count the number of trees in each cell
xmin, ymin, xmax, ymax = als_trees_in_boundary.total_bounds
cell_size = 10

grid_cells = []
for x0 in np.arange(xmin, xmax+cell_size, cell_size ):
    for y0 in np.arange(ymin, ymax+cell_size, cell_size):
        # bounds
        x1 = x0-cell_size
        y1 = y0+cell_size
        grid_cells.append( shapely.geometry.box(x0, y0, x1, y1)  )

cell = gpd.GeoDataFrame(grid_cells, columns=['geometry'], 
                                 crs=als_trees_in_boundary.crs)

cell['ind_name'] = np.arange(cell.shape[0])

count_trees = gpd.sjoin(als_trees_in_boundary, cell,  how='left', predicate='within')
count_trees['n_trees'] = 1
count_trees['trees'] = 0

count_trees['n_trees'] = count_trees['n_trees'].fillna(0)
count_trees['trees'] = count_trees['trees'].fillna(0)
count_trees['n_trees'] = count_trees['n_trees'] + count_trees['trees']
count_trees['n_trees'] = count_trees.n_trees.replace(0, np.nan)

dissolved = count_trees.dissolve(by = 'index_right', aggfunc="count")

cell.loc[dissolved.index, 'n_trees'] = dissolved.n_trees.values

cell['proportions'] = cell.n_trees/np.nansum(cell.n_trees)
cell_na_remove = cell.dropna()
cell.to_file(project_folder + 'data/cell_tree_density.geojson')

#adds the correct number of trees to places where there are a lot of trees, adds it to a weighted random grid cell
where_to_add = random.choices(population = np.array(cell_na_remove['ind_name']), 
               weights = np.array(cell_na_remove['n_trees']), 
               k = int(trees_to_add))

#get the random x y location in the grid cell
loc_in_grid = []
for i in np.arange(len(where_to_add)):
    loc_in_grid.append(cell[cell.ind_name == where_to_add[i]].sample_points(size=1).reset_index(drop = True)[0])


als_trees_simple = pd.DataFrame({
    'HT':als_trees_in_boundary.HT,
    'x':als_trees_in_boundary.geometry.x,
    'y':als_trees_in_boundary.geometry.y,
})

new_trees  = als_trees_simple

new_trees_gdf = gpd.GeoDataFrame(new_trees, 
                                 geometry = gpd.points_from_xy(new_trees.x, new_trees.y), 
                                 crs = als_trees_in_boundary.crs)


2483.0


In [16]:
tree = '04_tree.csv'
# site_name = 'ind'
field_trees = pd.read_csv(plot_folder + tree)
plots_df = pd.read_csv(plot_folder + '01_plot_identification.csv')

plots_df = plots_df[plots_df.site_name == site_name]
field_trees = field_trees.merge(plots_df, on = 'inventory_id')
field_trees_crs = np.unique(field_trees.plot_coord_srs)[0]

field_trees['HT'] = field_trees.tree_ht
field_trees['DIA'] = field_trees.tree_dbh

field_trees['SCI_NAME'] = field_trees.tree_sp_scientific_name
field_trees['STATUSCD'] = np.where(field_trees.tree_status == 'Live', 1, 0)
field_trees['CR'] = (field_trees.tree_htlcb/field_trees.tree_ht) #* 100
field_trees['X'] = field_trees.plot_coord_x
field_trees['Y'] = field_trees.plot_coord_y


#add spcd
field_trees = field_trees.merge(tree_species_code, on = 'SCI_NAME')

#only keep the values that are important
field_trees = field_trees[['SPCD', 'STATUSCD', 'DIA', 'HT', 'CR', 'X', 'Y']]

#if CR is nan, replace with 0.6
field_trees = field_trees.fillna(value = {'CR': 0.6})
field_trees_gpd = gpd.GeoDataFrame(field_trees, 
                               geometry = gpd.points_from_xy(field_trees.X, field_trees.Y),
                              crs = field_trees_crs)

training_treelist = field_trees_gpd
training_treelist = training_treelist.to_crs(boundary.crs)

#generate random heights based on the field data distribution
mu, std = norm.fit(field_trees.HT)

generated_ht = []
for i in np.arange(len(where_to_add)):
    random_ht = -1
    
    while (random_ht < np.nanmin(field_trees.HT) or random_ht > np.nanmax(field_trees.HT)):
        random_ht = np.random.normal(mu, std)

    generated_ht.append(random_ht)

generated_ht_gdf = gpd.GeoDataFrame(pd.DataFrame({'HT': generated_ht}), 
                                    geometry = loc_in_grid, 
                                    crs = cell.crs)
generated_ht_gdf['x'] = generated_ht_gdf.geometry.x
generated_ht_gdf['y'] = generated_ht_gdf.geometry.y

new_trees_gdf = pd.concat([new_trees_gdf, generated_ht_gdf[new_trees_gdf.columns]])
midstory = pd.DataFrame()


In [17]:
training_treelist['CBH'] = training_treelist['CR'] * training_treelist['HT']

#predict DBH from simulated tree heights
training_treelist_bound = training_treelist.sjoin(boundary)

new_trees_gdf = new_trees_gdf.dropna(subset = ['HT'])
new_tree_ht = np.array(new_trees_gdf.HT).reshape(np.array(new_trees_gdf.HT).shape[0], 1)


In [18]:
#predict DBH from simulated tree heights, choose between linear or exponential fit depending on the field data
training_treelist_bound = training_treelist.sjoin(boundary)

new_trees_gdf = new_trees_gdf.dropna(subset = ['HT'])
new_tree_ht = np.array(new_trees_gdf.HT).reshape(np.array(new_trees_gdf.HT).shape[0], 1)

def exponential_func(x, a, b):
    return a * np.exp(b * x)


params, covariance = curve_fit(exponential_func, training_treelist_bound.HT, training_treelist_bound.DIA, p0=[1, 0.5])
a_fit, b_fit = params

y_predicted = exponential_func(training_treelist_bound.HT, a_fit, b_fit)
r2_exp = r2_score(training_treelist_bound.DIA, y_predicted)

X = np.array(training_treelist_bound.HT).reshape(np.array(training_treelist_bound.HT).shape[0],1)
Y = np.array(training_treelist_bound.DIA).reshape(np.array(training_treelist_bound.DIA).shape[0],1)

mod = LinearRegression()
mod_fit = mod.fit(X,Y)

y_predicted = mod.predict(X)
r2_lin = r2_score((training_treelist_bound.DIA), y_predicted)

if r2_exp > r2_lin:
    dbh = exponential_func(new_tree_ht, a_fit, b_fit)

else:
    dbh = mod.predict(np.array(new_tree_ht).reshape(np.array(new_tree_ht).shape[0],1))


In [19]:

new_trees_gdf['dbh'] = dbh
new_trees_gdf= new_trees_gdf.rename(columns = {'height':'HT', 'dbh':'DIA'})


# pointsxy = gpd.points_from_xy(fftl_bound.X, fftl_bound.Y, crs = fftl_bound.crs).to_crs(new_trees_gdf.crs)
pointsxy = training_treelist_bound
pointsxy['x'] = pointsxy.geometry.x
pointsxy['y'] = pointsxy.geometry.y

training_treelist_bound['x_point'] = pointsxy.x
training_treelist_bound['y_point'] = pointsxy.y

new_trees_gdf['x_point'] = new_trees_gdf.geometry.x
new_trees_gdf['y_point'] = new_trees_gdf.geometry.y

In [20]:

df_no_na = training_treelist_bound.dropna(subset = ['HT', 'DIA', 'SPCD','x_point', 'y_point'])


In [21]:

#calculate and add the slope and aspect to the training treelist
tif = rasterio.open(project_folder + 'data/dtm.tif')
meta = tif.meta

origin_x = meta['transform'][2] #origin is top left
origin_y = meta['transform'][5]
res = meta['transform'][0]

i_list = ((df_no_na.geometry.x.values - origin_x)/res)
j_list = ((origin_y - df_no_na.geometry.y.values)/res)

slope_val = []
aspect_val = []

for n in np.arange(np.shape(i_list)[0]):
    slp = slope_map[int(j_list[n]), int(i_list[n])]
    asp = aspect_map[int(j_list[n]), int(i_list[n])]

    if np.isnan(slp):
        slope_val.append(np.nan)
    else:
        slope_val.append(round(slp))

    if np.isnan(asp):
        aspect_val.append(np.nan)
    else:
        aspect_val.append(round(asp))
        

df_no_na['slope'] = slope_val
df_no_na['aspect'] = aspect_val

#get rid of trees that aren't in the area

min_x = origin_x
max_x = min_x + (meta['width'] * res)

max_y = origin_y
min_y = max_y - (meta['height'] * res)

new_trees_gdf = new_trees_gdf[new_trees_gdf.x > min_x]
new_trees_gdf = new_trees_gdf[new_trees_gdf.x < max_x]
new_trees_gdf = new_trees_gdf[new_trees_gdf.y < max_y]
new_trees_gdf = new_trees_gdf[new_trees_gdf.y > min_y]

#calculate and add the slope and aspect to the new trees treelist

i_list = ((new_trees_gdf.geometry.x.values - origin_x)/res)
j_list = ((origin_y - new_trees_gdf.geometry.y.values)/res)

slope_val = []
aspect_val = []

for n in np.arange(np.shape(i_list)[0]):
    slp = slope_map[int(j_list[n]), int(i_list[n])]
    asp = aspect_map[int(j_list[n]), int(i_list[n])]

    if np.isnan(slp):
        slope_val.append(np.nan)
    else:
        slope_val.append(round(slp))

    if np.isnan(asp):
        aspect_val.append(np.nan)
    else:
        aspect_val.append(round(asp))
        
new_trees_gdf['slope'] = slope_val
new_trees_gdf['aspect'] = aspect_val

new_trees_gdf = new_trees_gdf.dropna()
new_trees_gdf['X'] = new_trees_gdf['x']
new_trees_gdf['Y'] = new_trees_gdf['y']


In [22]:
target = 'SPCD'
training = df_no_na
training_columns = ['HT', 'DIA', 'slope', 'aspect']

training = training.dropna(subset = ['SPCD', 'HT', 'DIA'])


X = df_no_na[training_columns]
y = df_no_na[[target]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)
y_train2 = np.array(y_train).reshape(np.array(y_train).shape[0], )
x_predict = new_trees_gdf[training_columns]

In [23]:

#predict species based on predicted dbh and height

clf = RandomForestClassifier(
    n_estimators=317,
    max_depth=23,
    min_samples_split=6,
    min_samples_leaf=1,
    max_features=None,
    bootstrap=True,
    criterion='log_loss',
    class_weight='balanced',
    random_state=42
)

clf.fit(X_train, y_train2)
pred = clf.predict(X_test)
score = clf.score(X_test, y_test)
predicted_species = clf.predict(x_predict)

#clean up species data frames
training_treelist_species = training_treelist.merge(tree_species_code, on = 'SPCD')

new_trees_gdf['SPCD'] = predicted_species

#assume all trees are alive
new_trees_gdf['STATUSCD'] = 1


In [25]:
#estimate crown ratio from dbh and crown base height from height and dbh
fftl = training_treelist_species
for n, s in enumerate(np.unique(new_trees_gdf.SPCD)):
    subset_df = fftl[fftl.SPCD == s]
    subset_df['dbh'] = subset_df.DIA# np.log(subset_df.DIA_cm)
     
    if subset_df.shape[0] > 0:
        new_trees_gdf_subset = new_trees_gdf[new_trees_gdf.SPCD == s]
         
        #estimating crown ratio from dbh
        predict_x =  np.array(new_trees_gdf_subset.DIA).reshape(np.array(new_trees_gdf_subset.DIA).shape[0],1)
         
        # est_crownratio = lin_reg(subset_df,  'DIA_cm', 'CROWN_RADIUS_m',predict_x)/100
        est_crownratio = np.array([i for i in lin_reg(subset_df,  'DIA', 'CR',predict_x)])#/100
        new_trees_gdf_subset['CR'] = est_crownratio
         
        #estimating crown base height
        subset_df = subset_df[['DIA', 'HT', 'CBH']].dropna()
         
        X = np.array(subset_df[['DIA', 'HT']]).reshape(-1,2)
        y = np.array(subset_df['CBH'])#.reshape(-1,1)
         
        predict_x =  np.array(new_trees_gdf_subset[['DIA', 'HT']]).reshape(-1,2)
         
        cbh_lin = LinearRegression()
        cbh_lin.fit(X,y)
             
        predicted_cbh = cbh_lin.predict(predict_x)
        new_trees_gdf_subset['CBH'] = predicted_cbh #this part gives a warning
         
        if n == 0:
            new_trees_with_extra = new_trees_gdf_subset
        else:
            new_trees_with_extra = pd.concat([new_trees_with_extra, new_trees_gdf_subset])
     
    else:
        new_trees_gdf_subset = new_trees_gdf[new_trees_gdf.SPCD == s]
        new_trees_gdf_subset['CR'] = new_trees_gdf_subset.HT / 10
        new_trees_gdf_subset['CBH'] = new_trees_gdf_subset.HT / 2
         
        if n == 0:
            new_trees_with_extra = new_trees_gdf_subset
        else:
            new_trees_with_extra = pd.concat([new_trees_with_extra, new_trees_gdf_subset])

#clean up data frame
new_trees_all = new_trees_with_extra
new_trees_all = new_trees_all[new_trees_all.DIA > 0]
# new_trees_all = new_trees_all.drop('index', axis = 1).reset_index()
# new_trees_all = new_trees_all.drop('index', axis = 1)

#add in Genus and Species
new_trees_all = new_trees_all.merge(tree_species_code[['SPCD','SCI_NAME']], on = 'SPCD')

new_trees_all['Genus'] = [str.split(s, ' ')[0] for s in new_trees_all.SCI_NAME]
new_trees_all['Species'] = [str.split(s, ' ')[1] for s in new_trees_all.SCI_NAME]

#more cleaning up
new_trees_all = new_trees_all.to_crs(5070)
new_trees_all['X'] = new_trees_all.geometry.x
new_trees_all['Y'] = new_trees_all.geometry.y

new_trees_all = new_trees_all[new_trees_all.DIA > 0]
new_trees_all = new_trees_all[new_trees_all.DIA < 1000]

#set max height to not be taller than the fftl
new_trees_all = new_trees_all[new_trees_all.HT < (np.nanmax(fftl.HT) + 10)]
new_trees_all['TREE_ID'] = np.arange(new_trees_all.shape[0])


columns_of_interest = ['TREE_ID', 'SPCD', 'STATUSCD', 'DIA', 'HT', 'CR','X', 'Y', 'CBH']


/opt/anaconda3/envs/gdal_env3/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/anaconda3/envs/gdal_env3/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/opt/anaconda3/envs/gdal_env3/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Da

In [26]:
als_modified_treelist = new_trees_all[columns_of_interest]

In [33]:
als_modified_treelist['CR'] = als_modified_treelist['CR'].mask(als_modified_treelist['CR'] > 1 , 1)

#new treelist
save_mod_treelist = project_folder + 'data/treelist_mod.csv'

#fix the areas of where the trees are

als_modified_treelist.to_csv(save_mod_treelist)


/var/folders/rp/zdr20wpx60j7227xn_x7mjqc0000gn/T/ipykernel_6274/3257697588.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  als_modified_treelist['CR'] = als_modified_treelist['CR'].mask(als_modified_treelist['CR'] > 1 , 1)


In [ ]:
import os

os.environ["FASTFUELS_API_KEY"] = "test-api-key"

TREELIST_CSV = save_mod_treelist

# ── Domain ───────────────────────────────────────────────────────────────────
print("Creating domain...")
domain = Domain.from_geodataframe(
    geodataframe=boundary,
    name="innovation factory",
    description="innovation factory with field data",
    horizontal_resolution=2.0,
    vertical_resolution=1.0,
)
domain_id = domain.id
print(f"Domain: {domain.name} ({domain_id})")

# ── Tree Inventory ───────────────────────────────────────────────────────────
print(f"\nUploading tree inventory...")
tree_inventory = Inventories.from_domain_id(domain_id).create_tree_inventory_from_file_upload(
    TREELIST_CSV
)
tree_inventory = tree_inventory.wait_until_completed(verbose=False)
print(f"Tree inventory status: {tree_inventory.status}")

# ── Tree Inventory Export ────────────────────────────────────────────────────
print("\nExporting FastFuels Trees to geojson...")
tree_inventory_export = tree_inventory.create_export("geojson")
tree_inventory_export = tree_inventory_export.wait_until_completed(verbose=False, in_place=True)
tree_inventory_export.to_file(gui_folder + "fastfuels_tree_inventory.geojson")
print(f"Exported tree inventory to data/fastfuels_tree_inventory.geojson")

# ── Tree Grid ────────────────────────────────────────────────────────────────
print("\nBuilding tree grid...")
tree_grid = (
    TreeGridBuilder(domain_id=domain_id)
    .with_bulk_density_from_tree_inventory()
    .with_spcd_from_tree_inventory()
    .build()
)
tree_grid = tree_grid.wait_until_completed(verbose=False, in_place=True)
print(f"Tree grid status: {tree_grid.status}")

# ── Grid Export ───────────────────────────────────────────────────────────────
print("\nExporting grids to zarr...")
grid_export = Grids.from_domain_id(domain_id).create_export("zarr")
grid_export = grid_export.wait_until_completed(verbose=False, in_place=True)

# Save the data
output_path = gui_folder +  "canopy_fuel_grid_field_data.zip"
grid_export.to_file(str(output_path))
print(f"\nExported to {output_path}")


Creating domain...
Domain: innovation factory (ea05b630ba0c4aa9b4906a39ee9b17ac)

Uploading tree inventory...
Tree inventory status: JobStatus.COMPLETED

Exporting FastFuels Trees to geojson...
Exported tree inventory to data/fastfuels_tree_inventory.geojson

Building tree grid...
Tree grid status: JobStatus.COMPLETED

Exporting grids to zarr...


In [ ]:
import zarr

z = zarr.open(output_path)
fastfuels_tree_voxelized = z['tree']['bulkDensity'][...]

In [ ]:
import matplotlib.pyplot as plt
import copy
import numpy as np

data = fastfuels_tree_voxelized
masked_data = np.ma.masked_where(data <= 0.01, data) # Mask all zeros

# masked_data = np.flipud(masked_data)

current_cmap = copy.copy(plt.cm.viridis)
current_cmap.set_bad(color='white')


fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(np.flipud(masked_data.mean(axis=0)), cmap=current_cmap)
ax.set_title("Mean Bulk Density")
ax.set_xlabel("X")
ax.set_ylabel("Y")
fig.colorbar(im, ax=ax, label="Bulk Density (kg/m^3)", orientation="horizontal")
plt.tight_layout()
plt.show()